In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore') # Keeps the notebook clean for the presentation

In [2]:
def load_npz_to_dataframe(filepath):
    """Loads the .npz logs and extracts the exact telemetry and completion flags."""
    data = np.load(filepath)

    df = pd.DataFrame({
        'timestamps': data['timestamps'],
        'x': data['x'],
        'y': data['y'],
        'velocity': data['velocity'],
        'steering_angle': data['steering_angle'],
        'collision_flag': data['collision_flag'],
        'completion_flag': data['completion_flag']
    })

    # Standardize time to start at 0 seconds
    df['elapsed_time'] = df['timestamps'] - df['timestamps'].iloc[0]

    # Calculate cumulative distance to align paths spatially
    dx = np.diff(df['x'], prepend=df['x'].iloc[0])
    dy = np.diff(df['y'], prepend=df['y'].iloc[0])
    df['step_distance'] = np.sqrt(dx**2 + dy**2)
    df['cumulative_distance'] = df['step_distance'].cumsum()

    return df

In [3]:
def calculate_lap_metrics(df_mpc, df_lstm):
    mpc_time = df_mpc['elapsed_time'].iloc[-1]
    lstm_time = df_lstm['elapsed_time'].iloc[-1]

    print("🏁 LAP TIME DIFFERENCE")
    print("-" * 30)
    print(f"Baseline (MPC): {mpc_time:.3f} seconds")
    print(f"Model (LSTM):   {lstm_time:.3f} seconds")
    print(f"Difference:     {abs(mpc_time - lstm_time):.3f} seconds\n")

    print("🚦 LAP COMPLETION STATUS")
    print("-" * 30)
    # Check MPC Baseline
    if df_mpc['completion_flag'].max() > 0:
        print("MPC Baseline: ✅ Completed the lap successfully.")
    elif df_mpc['collision_flag'].max() > 0:
        print("MPC Baseline: 💥 CRASHED into a wall.")
    else:
        print("MPC Baseline: ⚠️ Did not finish (Time ran out).")

    # Check LSTM Model
    if df_lstm['completion_flag'].max() > 0:
        print("LSTM Model:   ✅ Completed the lap successfully.")
    elif df_lstm['collision_flag'].max() > 0:
        print("LSTM Model:   💥 CRASHED into a wall.")
    else:
        print("LSTM Model:   ⚠️ Did not finish (Time ran out).")

def calculate_spatial_similarity(df_mpc, df_lstm):
    max_dist = min(df_mpc['cumulative_distance'].max(), df_lstm['cumulative_distance'].max())
    shared_distances = np.linspace(0, max_dist, num=1000)

    mpc_x_func = interp1d(df_mpc['cumulative_distance'], df_mpc['x'], kind='linear')
    mpc_y_func = interp1d(df_mpc['cumulative_distance'], df_mpc['y'], kind='linear')

    lstm_x_func = interp1d(df_lstm['cumulative_distance'], df_lstm['x'], kind='linear')
    lstm_y_func = interp1d(df_lstm['cumulative_distance'], df_lstm['y'], kind='linear')

    spatial_errors = np.sqrt((mpc_x_func(shared_distances) - lstm_x_func(shared_distances))**2 +
                             (mpc_y_func(shared_distances) - lstm_y_func(shared_distances))**2)

    print("\n📏 PATH SIMILARITY")
    print("-" * 30)
    print(f"Average Path Deviation: {np.mean(spatial_errors):.3f} meters")
    print(f"Maximum Path Deviation: {np.max(spatial_errors):.3f} meters")

In [4]:
def plot_comparisons(df_mpc, df_lstm, map_name="Track"):
    fig, axs = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle(f'Trajectory and Control Comparison: {map_name}', fontsize=16, fontweight='bold')

    # 1. Trajectory (X vs Y)
    axs[0].plot(df_mpc['x'], df_mpc['y'], label='Baseline (MPC)', color='#2c3e50', linewidth=2)
    axs[0].plot(df_lstm['x'], df_lstm['y'], label='Model (LSTM)', color='#e74c3c', linestyle='--', linewidth=2)
    axs[0].set_title('Racing Line Path')
    axs[0].set_xlabel('X Position (m)')
    axs[0].set_ylabel('Y Position (m)')
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    # 2. Velocity
    axs[1].plot(df_mpc['elapsed_time'], df_mpc['velocity'], label='MPC', color='#2c3e50', linewidth=2)
    axs[1].plot(df_lstm['elapsed_time'], df_lstm['velocity'], label='LSTM', color='#e74c3c', linestyle='--', linewidth=2)
    axs[1].set_title('Velocity Profile')
    axs[1].set_xlabel('Elapsed Time (s)')
    axs[1].set_ylabel('Speed (m/s)')
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)

    # 3. Steering Angle
    axs[2].plot(df_mpc['elapsed_time'], df_mpc['steering_angle'], label='MPC', color='#2c3e50', linewidth=2)
    axs[2].plot(df_lstm['elapsed_time'], df_lstm['steering_angle'], label='LSTM', color='#e74c3c', linestyle='--', linewidth=2)
    axs[2].set_title('Steering Angle Profile')
    axs[2].set_xlabel('Elapsed Time (s)')
    axs[2].set_ylabel('Angle (rad)')
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# UPDATE THESE TO MATCH THE FILES YOU DOWNLOADED FROM DRIVE
MPC_DATA_PATH = "/home/devin_work/work/f1tenth/ApproxiMPC/LSTM_training/data/raceline_validation_data/MPC (gymkhana)/stmpc_IMS_normal.npz"
LSTM_DATA_PATH = "/home/devin_work/work/f1tenth/ApproxiMPC/LSTM_training/data/raceline_validation_data/LSTM (ROS)/curvature lstm/curvature_lstm_logs/IMS/seqlstm_128_IMS.npz"

KeyError: 'timestamps is not a file in the archive'

In [ ]:
# Load and process
mpc_df = load_npz_to_dataframe(MPC_DATA_PATH)
lstm_df = load_npz_to_dataframe(LSTM_DATA_PATH)

# Display Metrics
calculate_lap_metrics(mpc_df, lstm_df)
calculate_spatial_similarity(mpc_df, lstm_df)

# Display Charts
plot_comparisons(mpc_df, lstm_df, map_name="")